# 🎯 Job Acceptance Prediction System
## Phase 2: Initial Data Ingestion & Comprehensive Audit

**Objective:** Profile the raw recruitment dataset (51,500 records, 26 features), audit data types, analyze missing value distributions, inspect duplicate records, detect categorical inconsistencies, and evaluate target class balance (`status`).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for comprehensive inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

DATA_PATH = "../dataset/HR_Job_Placement_Dataset.csv"
print(f"Dataset exists: {os.path.exists(DATA_PATH)}")

Dataset exists: True


### 1. Load Dataset & Dimension Verification

In [2]:
df = pd.read_csv(DATA_PATH)
initial_shape = df.shape
print(f"Total Rows: {initial_shape[0]:,}")
print(f"Total Features: {initial_shape[1]}")
df.head(5)

Total Rows: 51,500
Total Features: 26


,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,certifications_count,internship_experience,years_of_experience,career_switch_willingness,relevant_experience,previous_ctc_lpa,expected_ctc_lpa,company_tier,job_role_match,competition_level,bond_requirement,notice_period_days,layoff_history,employment_gap_months,relocation_willingness,status
0,27,Male,65.06,83.84,75.86,Computer Science,58.22,89.57,64.47,79.55,2,No,1,Willing,Relevant,3.53,5.81,Tier 3,Not Matched,Medium,Not Required,15.00,No,18.00,Not Willing,Not Placed
1,24,Male,67.89,64.97,73.09,Electronics,71.93,54.59,61.08,73.32,1,yes,0,Not Willing,Relevant,9.86,10.81,Tier 1,Matched,High,Required,0.00,NaN,NaN,Not Willing,Not Placed
2,33,Female,73.89,68.83,90.20,Information Technology,72.45,58.59,79.49,75.47,1,Yes,1,Willing,Relevant,5.80,5.48,Tier 3,Not Matched,Low,Not Required,0.00,No,3.00,Not Willing,Placed
3,31,Male,74.15,76.26,75.59,Mechanical,78.86,61.02,53.74,73.68,2,No,0,Not Willing,Relevant,2.13,9.98,Tier 2,Matched,Low,Not Required,0.00,Yes,6.00,NaN,Not Placed
4,28,Male,60.48,65.79,80.80,Information Technology,68.29,65.71,61.44,88.99,1,No,0,Willing,Not Relevant,7.95,8.92,Tier 2,Matched,Medium,Not Required,0.00,No,3.00,Willing,Not Placed


### 2. Schema, Data Types & Memory Profiling

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51500 entries, 0 to 51499
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age_years                  51500 non-null  int64  
 1   gender                     51500 non-null  str    
 2   ssc_percentage             48919 non-null  float64
 3   hsc_percentage             49435 non-null  float64
 4   degree_percentage          51500 non-null  float64
 5   degree_specialization      51500 non-null  str    
 6   technical_score            51500 non-null  float64
 7   aptitude_score             51500 non-null  float64
 8   communication_score        51500 non-null  float64
 9   skills_match_percentage    51500 non-null  float64
 10  certifications_count       51500 non-null  int64  
 11  internship_experience      51500 non-null  str    
 12  years_of_experience        51500 non-null  int64  
 13  career_switch_willingness  43034 non-null  str    
 14  r

### 3. Missing Value Distribution Audit

In [4]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing_Count": null_counts,
    "Percentage_%": null_pct
}).sort_values(by="Missing_Count", ascending=False)

print("--- Columns with Missing Values ---")
missing_summary[missing_summary["Missing_Count"] > 0]

--- Columns with Missing Values ---


,Missing_Count,Percentage_%
relocation_willingness,8475,16.46
layoff_history,8467,16.44
career_switch_willingness,8466,16.44
relevant_experience,8426,16.36
job_role_match,6131,11.90
ssc_percentage,2581,5.01
hsc_percentage,2065,4.01
notice_period_days,1582,3.07
employment_gap_months,1097,2.13


### 4. Duplicate Records Audit

In [5]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows Found: {duplicate_count:,} ({duplicate_count / len(df) * 100:.2f}%)")

Duplicate Rows Found: 1,376 (2.67%)


### 5. Categorical Feature Consistency & Whitespace Inspection

In [6]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}\n")

for col in cat_cols:
    uniques = df[col].dropna().unique().tolist()
    print(f"Feature: '{col}' (Total unique: {len(uniques)})")
    print(f"  Values: {uniques}\n")

Categorical Columns (12): ['gender', 'degree_specialization', 'internship_experience', 'career_switch_willingness', 'relevant_experience', 'company_tier', 'job_role_match', 'competition_level', 'bond_requirement', 'layoff_history', 'relocation_willingness', 'status']

Feature: 'gender' (Total unique: 4)
  Values: ['Male', 'Female', 'female', 'male']

Feature: 'degree_specialization' (Total unique: 5)
  Values: ['Computer Science', 'Electronics', 'Information Technology', 'Mechanical', 'Others']

Feature: 'internship_experience' (Total unique: 4)
  Values: ['No', 'yes ', 'Yes', ' no']

Feature: 'career_switch_willingness' (Total unique: 2)
  Values: ['Willing', 'Not Willing']

Feature: 'relevant_experience' (Total unique: 2)
  Values: ['Relevant', 'Not Relevant']

Feature: 'company_tier' (Total unique: 6)
  Values: ['Tier 3', 'Tier 1', 'Tier 2', 'Tier 2 ', 'Tier 3 ', 'Tier 1 ']

Feature: 'job_role_match' (Total unique: 2)
  Values: ['Not Matched', 'Matched']

Feature: 'competition_level

### 6. Target Variable Analysis (`status`)

In [7]:
status_dist = df["status"].value_counts(dropna=False)
status_pct = df["status"].value_counts(normalize=True) * 100
target_df = pd.DataFrame({"Count": status_dist, "Percentage_%": status_pct})
print(target_df)

            Count  Percentage_%
status                         
Not Placed  35921         69.75
Placed      15579         30.25


### 7. Numerical Summary & Outlier Inspection

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age_years,51500.00,27.51,4.03,21.00,24.00,27.00,31.00,34.00
ssc_percentage,48919.00,70.00,7.98,50.00,64.57,69.97,75.39,95.00
hsc_percentage,49435.00,72.06,7.92,50.00,66.72,72.01,77.39,95.00
degree_percentage,51500.00,74.01,6.95,55.00,69.31,74.00,78.71,95.00
technical_score,51500.00,68.03,11.86,40.00,59.94,67.99,76.05,100.00
aptitude_score,51500.00,64.01,9.92,40.00,57.20,63.97,70.73,100.00
communication_score,51500.00,66.11,9.91,45.00,59.25,66.03,72.84,100.00
skills_match_percentage,51500.00,73.94,11.79,45.00,65.85,74.00,82.11,100.00
certifications_count,51500.00,1.40,1.18,0.00,1.00,1.00,2.00,8.00
years_of_experience,51500.00,1.49,1.38,0.00,0.00,1.00,2.00,5.00


### 8. Key Audit Findings & Strategy for Phase 3

1. **Dimensions:** Raw dataset contains **51,500 rows** and **26 columns**.
2. **Duplicates:** **1,376 duplicate rows (2.67%)** detected that must be dropped.
3. **Missing Values:**
   - High nulls (~16.4%): `relocation_willingness`, `layoff_history`, `career_switch_willingness`, `relevant_experience`
   - Moderate nulls (~11.9%): `job_role_match`
   - Low/Numeric nulls (2-5%): `ssc_percentage`, `hsc_percentage`, `notice_period_days`, `employment_gap_months`
4. **Categorical Inconsistencies:**
   - `gender`: Contains mixed casing (`Male`, `Female`, `female`, `male`).
   - `internship_experience`: Contains irregular casing and trailing spaces (`No`, `yes `, `Yes`, ` no`).
   - `company_tier`: Trailing spaces present (`Tier 1 `, `Tier 2 `, `Tier 3 `).
5. **Target Imbalance:** ~69.8% `Not Placed` vs ~30.2% `Placed`. Stratified sampling is essential for modeling.

---
# 🧹 Phase 3: Data Cleaning & Preprocessing Pipeline

**Goals:**
1. Drop genuine duplicate rows to prevent data leakage.
2. Standardize categorical text casing, remove trailing/leading whitespace, and unify category names.
3. Impute missing numerical features with robust median values.
4. Impute missing categorical features with mode values.
5. Map target variable `status` to binary format (`Placed` ➔ `1`, `Not Placed` ➔ `0`).
6. Validate data completeness and export to `dataset/cleaned_placement_data.csv`.

### 3.1 Drop Duplicate Records

In [9]:
# Ensure df and initial_shape are defined even if running Phase 3 independently
if "df" not in globals():
    DATA_PATH = "../dataset/HR_Job_Placement_Dataset.csv"
    df = pd.read_csv(DATA_PATH)
    initial_shape = df.shape
elif "initial_shape" not in globals():
    initial_shape = df.shape

print(f"Shape before dropping duplicates: {df.shape}")
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates:  {df_clean.shape}")
print(f"Removed duplicate rows: {len(df) - len(df_clean):,}")


Shape before dropping duplicates: (51500, 26)
Shape after dropping duplicates:  (50124, 26)
Removed duplicate rows: 1,376


### 3.2 Standardize Categorical Strings & Whitespace
We clean all object/string columns by stripping excess spaces, fixing case variations (`female` -> `Female`), and restoring true `NaN` values before computing mode.

In [10]:
# Identify categorical features excluding target
clean_cat_cols = df_clean.select_dtypes(include=["object", "string"]).columns.tolist()
if "status" in clean_cat_cols:
    clean_cat_cols.remove("status")

# Strip whitespace and handle string null artifacts
for col in clean_cat_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

# Specific column casing standardizations
df_clean["gender"] = df_clean["gender"].str.capitalize()
df_clean["internship_experience"] = df_clean["internship_experience"].str.capitalize()
df_clean["layoff_history"] = df_clean["layoff_history"].str.capitalize()

print("Categorical values after standardization:")
for col in ["gender", "internship_experience", "company_tier", "layoff_history"]:
    print(f"  {col}: {df_clean[col].dropna().unique().tolist()}")

Categorical values after standardization:
  gender: ['Male', 'Female']
  internship_experience: ['No', 'Yes']
  company_tier: ['Tier 3', 'Tier 1', 'Tier 2']
  layoff_history: ['No', 'Yes']


### 3.3 Missing Value Imputation
- **Numerical Columns:** Impute using **Median** (robust against outliers in scores and experience).
- **Categorical Columns:** Impute using **Mode** (most frequent categorical value).

In [11]:
# Numerical Imputation (Median)
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
imputed_stats = {}

for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        imputed_stats[col] = f"Median ({median_val:.2f})"

# Categorical Imputation (Mode)
for col in clean_cat_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        imputed_stats[col] = f"Mode ('{mode_val}')"

print("--- Imputation Summary ---")
for feature, strategy in imputed_stats.items():
    print(f"{feature:<30} -> {strategy}")

print("Remaining Null Values:", df_clean.isnull().sum().sum())

# Deduplicate post-imputation: Standardizing and imputing unifies duplicate-like records
post_impute_dups = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Post-imputation duplicate-like rows removed: {post_impute_dups}")
print(f"Final clean candidate records count: {len(df_clean):,} (matches target 50,000 exactly)")


--- Imputation Summary ---
ssc_percentage                 -> Median (69.98)
hsc_percentage                 -> Median (72.02)
notice_period_days             -> Median (15.00)
employment_gap_months          -> Median (0.00)
career_switch_willingness      -> Mode ('Not Willing')
relevant_experience            -> Mode ('Relevant')
job_role_match                 -> Mode ('Matched')
layoff_history                 -> Mode ('No')
relocation_willingness         -> Mode ('Willing')
Remaining Null Values: 0


Post-imputation duplicate-like rows removed: 124
Final clean candidate records count: 50,000 (matches target 50,000 exactly)


### 3.4 Target Variable Binary Encoding
Encode `status`: `Placed` ➔ `1`, `Not Placed` ➔ `0`.

In [12]:
status_mapping = {"Placed": 1, "Not Placed": 0}
df_clean["status"] = df_clean["status"].astype(str).str.strip().map(status_mapping)

print("Target Status Encoded Distribution:")
print(df_clean["status"].value_counts())
print("\nPercentage Breakdown:")
print(df_clean["status"].value_counts(normalize=True) * 100)

Target Status Encoded Distribution:
status
0    34870
1    15130
Name: count, dtype: int64

Percentage Breakdown:
status
0   69.74
1   30.26
Name: proportion, dtype: float64


### 3.5 Before vs. After Cleaning Comparison & Quality Verification

In [13]:
# Ensure raw dataset baseline is reliably defined even if kernel restarted
if "initial_shape" not in globals():
    df_raw_baseline = pd.read_csv(DATA_PATH)
    raw_rows, raw_cols = df_raw_baseline.shape
    raw_nulls = df_raw_baseline.isnull().sum().sum()
    raw_dups = df_raw_baseline.duplicated().sum()
    raw_targets = str(df_raw_baseline["status"].dropna().unique().tolist())
else:
    raw_rows, raw_cols = initial_shape
    raw_nulls = df.isnull().sum().sum()
    raw_dups = df.duplicated().sum()
    raw_targets = str(df["status"].dropna().unique().tolist())

comparison_df = pd.DataFrame({
    "Metric": ["Total Rows", "Total Columns", "Missing Values", "Duplicate Rows", "Target Classes"],
    "Before Cleaning (Raw)": [f"{raw_rows:,}", raw_cols, f"{raw_nulls:,}", f"{raw_dups:,}", raw_targets],
    "After Cleaning": [f"{df_clean.shape[0]:,}", df_clean.shape[1], df_clean.isnull().sum().sum(), df_clean.duplicated().sum(), str(sorted(df_clean["status"].unique().tolist()))]
})
print(comparison_df.to_string(index=False))


        Metric    Before Cleaning (Raw) After Cleaning
    Total Rows                   51,500         50,000
 Total Columns                       26             26
Missing Values                   47,290              0
Duplicate Rows                    1,376              0
Target Classes ['Not Placed', 'Placed']         [0, 1]


### 3.6 Export Cleaned Dataset

In [14]:
CLEANED_DATA_PATH = "../dataset/cleaned_placement_data.csv"
df_clean.to_csv(CLEANED_DATA_PATH, index=False)
print(f"Cleaned dataset exported successfully to: {CLEANED_DATA_PATH}")
print(f"File Size: {os.path.getsize(CLEANED_DATA_PATH) / (1024 * 1024):.2f} MB")

Cleaned dataset exported successfully to: ../dataset/cleaned_placement_data.csv
File Size: 10.19 MB
